# Lab 03: Multi-step research agent

Build a research agent that searches the real web, fetches pages, synthesizes
across sources, and cites what it actually read. From scratch — no framework,
no embeddings, no vector store. The agent has just two tools and a loop, but
the trajectory is non-trivial.

This is the runnable companion to
[`labs/03-multi-step-research-agent/README.md`](./README.md). Read the brief
first.

**Estimated time:** 90–120 minutes.
**Difficulty:** 🟡 Intermediate.
**Prerequisites:** Lab 01, Lab 02,
[`concepts/tools/search-tools.md`](../../concepts/tools/search-tools.md),
[`tools/search/snapshot-v1.0.md`](../../tools/search/snapshot-v1.0.md).

> 🔴 **Search libraries are fast-changing.** This notebook is pinned to
> `ddgs>=9.0,<10` (verified 2026-05-24). If something doesn't behave as
> described, check the snapshot first — search API surfaces shift faster
> than most other tooling.

## Step 0: Setup

Same provider-agnostic LLM client pattern as prior labs. Three new
dependencies for the real-web tools: `ddgs` for search, `requests` and
`beautifulsoup4` for page fetching.

```bash
uv add 'ddgs>=9.0,<10' 'beautifulsoup4>=4.12' 'requests>=2.31'
```

In [ ]:
import os
import pathlib
from dotenv import load_dotenv

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

PROVIDER = "openai"   # or "anthropic"

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), (
    "Set OPENAI_API_KEY or ANTHROPIC_API_KEY in .env"
)
print(f"Provider: {PROVIDER}")


**Sample output:**

```
Provider: openai
```

In [ ]:
# Provider-agnostic chat client. Same wrapper as Labs 01 and 02.

from typing import Any


def chat_with_tools(messages: list[dict], tools: list[dict],
                    model: str | None = None) -> dict:
    """Call the chat completion endpoint with tool schemas. Return the
    assistant message as a dict (role, content, tool_calls)."""
    if PROVIDER == "openai":
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model=model or "gpt-4o-mini",
            messages=messages,
            tools=tools,
            temperature=0,
        )
        msg = resp.choices[0].message
        return {
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {
                    "id": tc.id,
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                }
                for tc in (msg.tool_calls or [])
            ],
        }
    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        # Convert OpenAI-format tools to Anthropic format
        anthropic_tools = [
            {
                "name": t["function"]["name"],
                "description": t["function"]["description"],
                "input_schema": t["function"]["parameters"],
            }
            for t in tools
        ]
        # System message extraction (Anthropic separates it from messages)
        system = next((m["content"] for m in messages if m["role"] == "system"),
                      None)
        non_system = [m for m in messages if m["role"] != "system"]
        # Tool results need translation too — simplified for this lab
        resp = client.messages.create(
            model=model or "claude-haiku-4-5-20251001",
            system=system or "",
            messages=non_system,
            tools=anthropic_tools,
            max_tokens=2048,
        )
        content_blocks = resp.content
        text = "".join(b.text for b in content_blocks if hasattr(b, "text"))
        tool_calls = [
            {
                "id": b.id,
                "name": b.name,
                "arguments": _json_dumps(b.input),
            }
            for b in content_blocks
            if getattr(b, "type", None) == "tool_use"
        ]
        return {"role": "assistant", "content": text, "tool_calls": tool_calls}
    else:
        raise ValueError(f"Unknown provider: {PROVIDER}")


def _json_dumps(obj: Any) -> str:
    import json
    return json.dumps(obj)


print("LLM client ready.")


**Sample output:**

```
LLM client ready.
```

## Step 1: The `web_search` tool

Wrap `ddgs.text(...)` with the structured-error pattern from Lab 02.
The contract:

```python
web_search(query: str, recency: str = "any", max_results: int = 8) -> dict
```

Returns one of:

```python
{"status": "ok",     "results": [{"title": ..., "url": ..., "snippet": ...}, ...]}
{"status": "empty",  "query": ..., "detail": "no results"}
{"status": "error",  "kind": "rate_limit" | "timeout" | "other", "detail": ...}
```

Structured errors instead of exceptions: the agent can *read* the failure
and decide what to do, instead of crashing. This is the Lab 02 pattern
applied to a real backend.

In [ ]:
# The web_search tool
from typing import Literal

from ddgs import DDGS
from ddgs.exceptions import (
    DDGSException,
    RatelimitException,
    TimeoutException,
)

RecencyType = Literal["any", "day", "week", "month", "year"]

# Map our recency vocabulary to ddgs's timelimit codes
_RECENCY_MAP = {
    "any": None,
    "day": "d",
    "week": "w",
    "month": "m",
    "year": "y",
}


def web_search(query: str,
               recency: RecencyType = "any",
               max_results: int = 8) -> dict:
    """Search the web. Returns structured results or a structured error."""
    if not query or not query.strip():
        return {"status": "error", "kind": "other",
                "detail": "empty query"}

    timelimit = _RECENCY_MAP.get(recency)
    try:
        with DDGS(timeout=15) as ddgs:
            raw = ddgs.text(
                query=query.strip(),
                region="us-en",
                safesearch="moderate",
                timelimit=timelimit,
                max_results=max_results,
                backend="auto",
            )
    except RatelimitException as e:
        return {"status": "error", "kind": "rate_limit", "detail": str(e)}
    except TimeoutException as e:
        return {"status": "error", "kind": "timeout", "detail": str(e)}
    except DDGSException as e:
        return {"status": "error", "kind": "other", "detail": str(e)}
    except Exception as e:  # network errors, etc.
        return {"status": "error", "kind": "other",
                "detail": f"{type(e).__name__}: {e}"}

    if not raw:
        return {"status": "empty", "query": query,
                "detail": "no results returned"}

    # Normalize result shape — same keys we'd want if we swap to Tavily.
    normalized = [
        {
            "title": (r.get("title") or "").strip(),
            "url": (r.get("href") or "").strip(),
            "snippet": (r.get("body") or "").strip(),
        }
        for r in raw
        if r.get("href")  # drop malformed entries
    ]
    return {"status": "ok", "results": normalized[:max_results]}


# Quick smoke test (uncomment to run live; will hit the network)
# print(web_search("python programming language history", max_results=3))


**Sample output (when uncommented):**

```python
{
    "status": "ok",
    "results": [
        {
            "title": "History of Python - Wikipedia",
            "url": "https://en.wikipedia.org/wiki/History_of_Python",
            "snippet": "Python was conceived in the late 1980s by Guido van Rossum..."
        },
        ...
    ]
}
```

Note that we *catch* every specific `ddgs` exception type and return a
structured error dict instead. The agent sees a tool result, not a crash —
and can decide whether to retry, switch tactics, or surface the failure.

## Step 2: The `fetch_page` tool

The second tool fetches and cleans a single URL. Same structured-error
shape as `web_search`, but with new failure kinds: HTTP 4xx/5xx, paywall
detection (a content-length heuristic), and a `too_long` status that
flags truncation when a page exceeds `max_chars`.

Contract:

```python
fetch_page(url: str, max_chars: int = 8000) -> dict
```

Returns one of:

```python
{"status": "ok",       "url": ..., "title": ..., "text": ..., "elapsed_ms": ...}
{"status": "too_long", "url": ..., "title": ..., "text": ...[:max_chars], "truncated_at": int}
{"status": "error",    "url": ..., "kind": "timeout"|"http_4xx"|"http_5xx"|"blocked"|"parse"|"other", "detail": ...}
```

In [ ]:
# The fetch_page tool
import re
import time
import warnings

import requests
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning

# bs4 warns when text input looks like a URL/path — we feed it HTML, but
# some pages return short JS-redirect bodies that look like filenames.
# Silence the noise; it's cosmetic.
warnings.simplefilter("ignore", MarkupResemblesLocatorWarning)


# Politeness: identify ourselves so site operators can block if they want
USER_AGENT = (
    "AgenticAIEngineer-CourseLab/0.1 "
    "(https://github.com/MHHamdan/Agentic-AI-Engineer) "
    "Mozilla/5.0 (compatible)"
)


# Heuristic markers of paywall pages — appear in the body when content is gated
PAYWALL_MARKERS = [
    "subscribe to read",
    "subscribe to continue",
    "create a free account to continue",
    "you've reached your free article limit",
    "register to read",
]


def fetch_page(url: str, max_chars: int = 8000) -> dict:
    """Fetch a URL and return cleaned text, or a structured error."""
    if not url or not url.startswith(("http://", "https://")):
        return {"status": "error", "url": url, "kind": "other",
                "detail": "invalid url"}

    t0 = time.monotonic()
    try:
        resp = requests.get(
            url,
            headers={"User-Agent": USER_AGENT},
            timeout=15,
            allow_redirects=True,
        )
    except requests.Timeout:
        return {"status": "error", "url": url, "kind": "timeout",
                "detail": "request timed out after 15s"}
    except requests.ConnectionError as e:
        return {"status": "error", "url": url, "kind": "other",
                "detail": f"connection error: {e}"}
    except requests.RequestException as e:
        return {"status": "error", "url": url, "kind": "other",
                "detail": f"{type(e).__name__}: {e}"}

    elapsed_ms = int((time.monotonic() - t0) * 1000)

    # HTTP status handling
    if 400 <= resp.status_code < 500:
        kind = "blocked" if resp.status_code in (401, 403, 429) else "http_4xx"
        return {"status": "error", "url": url, "kind": kind,
                "detail": f"HTTP {resp.status_code}"}
    if 500 <= resp.status_code < 600:
        return {"status": "error", "url": url, "kind": "http_5xx",
                "detail": f"HTTP {resp.status_code}"}

    # Parse HTML — bs4's "html.parser" is stdlib-only and good enough here
    try:
        soup = BeautifulSoup(resp.text, "html.parser")
    except Exception as e:
        return {"status": "error", "url": url, "kind": "parse",
                "detail": f"{type(e).__name__}: {e}"}

    # Strip script, style, nav, footer, aside
    for tag in soup(["script", "style", "nav", "footer", "aside",
                     "header", "form", "iframe", "noscript"]):
        tag.decompose()

    title = (soup.title.string.strip() if soup.title and soup.title.string
             else "")

    # Extract visible text, then collapse whitespace
    text = soup.get_text(separator="\n")
    text = re.sub(r"\n\s*\n+", "\n\n", text)  # collapse blank lines
    text = re.sub(r"[ \t]+", " ", text).strip()

    # Paywall detection: short body + paywall marker
    text_lower = text.lower()
    if len(text) < 1500 and any(m in text_lower for m in PAYWALL_MARKERS):
        return {"status": "error", "url": resp.url, "kind": "blocked",
                "detail": "paywall or registration wall detected"}

    if len(text) > max_chars:
        return {
            "status": "too_long",
            "url": resp.url,
            "title": title,
            "text": text[:max_chars],
            "truncated_at": max_chars,
            "elapsed_ms": elapsed_ms,
        }

    return {
        "status": "ok",
        "url": resp.url,
        "title": title,
        "text": text,
        "elapsed_ms": elapsed_ms,
    }


# Smoke test on a known-stable URL (uncomment to run)
# r = fetch_page("https://example.com")
# print({k: (v[:80] if isinstance(v, str) else v) for k, v in r.items()})


**Sample output (when uncommented):**

```python
{
    "status": "ok",
    "url": "https://example.com",
    "title": "Example Domain",
    "text": "Example Domain\nThis domain is for use in illustrative examples...",
    "elapsed_ms": 142
}
```

## Step 3: Tool schemas + dispatcher

Two OpenAI-format tool schemas, and one `execute_tool(...)` dispatcher that
maps a tool call to the corresponding Python function. Same shape as Labs
01/02 — nothing new here except the *content* of the tools.

In [ ]:
import json

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": (
                "Search the web. Returns up to max_results items, each with "
                "title, url, and a short snippet. Use this to triage which "
                "pages are worth fetching. Use recency='week' or 'month' for "
                "news-like questions; leave as 'any' for stable knowledge."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query, 3-8 words usually best.",
                    },
                    "recency": {
                        "type": "string",
                        "enum": ["any", "day", "week", "month", "year"],
                        "description": (
                            "Time scope. Use 'any' for stable topics; "
                            "'week'/'month' for news; 'day' for breaking."
                        ),
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "1-10, default 8.",
                    },
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "fetch_page",
            "description": (
                "Fetch the full content of a single URL. Use when a snippet "
                "isn't enough to answer the question. Returns cleaned text; "
                "may return a structured error (timeout, http_4xx, blocked, "
                "paywall) or 'too_long' with truncation."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "url": {
                        "type": "string",
                        "description": "Absolute https:// or http:// URL.",
                    },
                    "max_chars": {
                        "type": "integer",
                        "description": "Truncate body to this length. Default 8000.",
                    },
                },
                "required": ["url"],
            },
        },
    },
]


def execute_tool(name: str, args: dict) -> dict:
    """Dispatch a tool call to the corresponding Python function."""
    if name == "web_search":
        return web_search(
            query=args["query"],
            recency=args.get("recency", "any"),
            max_results=args.get("max_results", 8),
        )
    if name == "fetch_page":
        return fetch_page(
            url=args["url"],
            max_chars=args.get("max_chars", 8000),
        )
    return {"status": "error", "kind": "other",
            "detail": f"unknown tool: {name}"}


print(f"Tools registered: {[t['function']['name'] for t in TOOLS]}")


**Sample output:**

```
Tools registered: ['web_search', 'fetch_page']
```

## Step 4: The agent loop

Lab 01's loop, plus three things this lab needs:

1. **Citation tracking.** Every time the agent calls `fetch_page` and gets
   an `ok`/`too_long` result, we record `{url, title}` in a `citations`
   list — separately from the LLM's working memory. This is the source of
   truth for what the agent *actually read*.
2. **Repeated-action detection.** We hash `(tool_name, args)` for each
   call and refuse to execute the same one twice. If the model tries
   anyway, we return a "repeated_action" error so it adjusts.
3. **Graceful step cap.** If we hit MAX_STEPS without a final answer, we
   exit with `stopped_reason="step_cap"` and surface what we *did* find,
   instead of pretending we have an answer.

In [ ]:
import hashlib

MAX_STEPS = 8

SYSTEM_PROMPT = """You are a research assistant. Answer the user's question
by searching the web and reading relevant pages.

Strategy:
1. Start with a web_search using 3-8 specific words from the question.
2. Look at the snippets. If they answer the question, synthesize directly.
   If not, pick the 1-2 most relevant URLs and call fetch_page on them.
3. After reading, if you still need more, search again with a refined query
   — but do not repeat queries you've already tried.
4. Stop when you can answer confidently with grounded evidence, OR after
   you have enough evidence that further searches won't help. Don't loop.
5. In your final answer, cite the URLs you actually read (the tool log
   will record these). Do not cite URLs you only saw in search results
   but did not open.

When you cannot find a confident answer, say so plainly. Do not guess or
fabricate facts. "I could not find a reliable answer" is an acceptable
response.

Use 'recency' to scope by freshness: 'week' or 'month' for current events,
'any' for stable knowledge.
"""


def _action_hash(name: str, args: dict) -> str:
    """Deterministic hash of a (tool_name, args) pair for dedup."""
    payload = name + "|" + json.dumps(args, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


def run_agent(question: str, max_steps: int = MAX_STEPS,
              verbose: bool = True) -> dict:
    """Run the research agent on a question. Return answer + citations."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    citations: list[dict] = []
    seen_actions: set[str] = set()

    for step in range(1, max_steps + 1):
        if verbose:
            print(f"\n── Step {step} ──")

        msg = chat_with_tools(messages, tools=TOOLS)

        # Append the assistant turn — OpenAI format expects tool_calls as
        # a list of dicts even when empty
        assistant_entry = {
            "role": "assistant",
            "content": msg["content"],
        }
        if msg["tool_calls"]:
            assistant_entry["tool_calls"] = [
                {
                    "id": tc["id"],
                    "type": "function",
                    "function": {
                        "name": tc["name"],
                        "arguments": tc["arguments"],
                    },
                }
                for tc in msg["tool_calls"]
            ]
        messages.append(assistant_entry)

        # If no tool calls, the assistant is giving a final answer
        if not msg["tool_calls"]:
            if verbose:
                print(f"  ◆ FINAL: {msg['content'][:120]}...")
            return {
                "answer": msg["content"],
                "citations": citations,
                "steps": step,
                "stopped_reason": "answer_with_citations" if citations
                else "answer_without_fetch",
            }

        # Execute each tool call
        for tc in msg["tool_calls"]:
            args = json.loads(tc["arguments"]) if tc["arguments"] else {}
            ah = _action_hash(tc["name"], args)

            # Repeated-action detection
            if ah in seen_actions:
                tool_result = {
                    "status": "error",
                    "kind": "repeated_action",
                    "detail": (
                        f"You already called {tc['name']} with these "
                        f"arguments. Try a different query or URL."
                    ),
                }
                if verbose:
                    print(f"  ✗ {tc['name']}({args}) [REPEATED — refused]")
            else:
                seen_actions.add(ah)
                tool_result = execute_tool(tc["name"], args)
                if verbose:
                    status = tool_result.get("status", "?")
                    args_repr = (str(args)[:80] + "...") if len(str(args)) > 80 \
                        else str(args)
                    print(f"  → {tc['name']}({args_repr}) → {status}")

                # Track citations: only for fetch_page successes
                if tc["name"] == "fetch_page" and tool_result.get("status") \
                        in ("ok", "too_long"):
                    citations.append({
                        "url": tool_result["url"],
                        "title": tool_result.get("title", ""),
                    })

            messages.append({
                "role": "tool",
                "tool_call_id": tc["id"],
                "content": json.dumps(tool_result)[:4000],  # cap tool result
            })

    # Step cap exit — graceful, surfaces what we found
    if verbose:
        print(f"\n  ⚠ Hit step cap ({max_steps}) without a final answer")
    return {
        "answer": (
            "I reached the step limit without a confident answer. "
            f"I fetched {len(citations)} page(s): "
            + ", ".join(c["url"] for c in citations[:3])
        ),
        "citations": citations,
        "steps": max_steps,
        "stopped_reason": "step_cap",
    }


print("Agent ready. Call run_agent('your question here').")


**Sample output:**

```
Agent ready. Call run_agent('your question here').
```

Two design notes worth pausing on:

- **Citations are recorded by our loop, not by the LLM.** This is a
  deliberate choice. If we asked the LLM "what URLs did you read?" at the
  end, it might fabricate one or hallucinate the title — both are common
  failure modes. Instead, our loop records the URL the moment
  `fetch_page` returns a successful result. The model can't add anything;
  it can't remove anything. That's what makes the citations trustworthy.

- **Repeated-action detection is on the *call*, not the *query*.** Two
  different `web_search` calls with different `recency` arguments are not
  considered duplicates — they're genuinely different searches. The
  `(name, args)` hash captures this.

## Step 5: Three test queries

Easy / Medium / Hard. Run them one at a time so the trajectory is readable.

> 💡 **First run is slow.** `ddgs`'s first call typically takes 5-15
> seconds while it picks a working backend. Subsequent calls are faster.
> This is the library, not your code.

In [ ]:
# EASY: a single search usually suffices. Maybe one fetch.
easy_q = "What is the Python ddgs library?"
print(f"QUERY: {easy_q}")
print("=" * 70)
result_easy = run_agent(easy_q, verbose=True)
print("=" * 70)
print(f"\n✓ Steps used: {result_easy['steps']}")
print(f"✓ Citations: {len(result_easy['citations'])}")
print(f"✓ Stopped reason: {result_easy['stopped_reason']}")
print(f"\nAnswer:\n{result_easy['answer']}")
print("\nCitations:")
for c in result_easy["citations"]:
    print(f"  - {c['url']}")


**Sample output (yours will vary — live web):**

```
QUERY: What is the Python ddgs library?
======================================================================

── Step 1 ──
  → web_search({'query': 'Python ddgs library', 'recency': 'any'}) → ok

── Step 2 ──
  → fetch_page({'url': 'https://pypi.org/project/ddgs/'}) → ok
  ◆ FINAL: The ddgs library is a Python metasearch library...
======================================================================

✓ Steps used: 2
✓ Citations: 1
✓ Stopped reason: answer_with_citations

Answer:
The ddgs (Dux Distributed Global Search) library is a Python metasearch
library that aggregates results from DuckDuckGo, Bing, Google, Brave,
and other search engines. It is published under MIT license, requires
Python 3.10+, and is maintained by deedy5 on GitHub. The library was
previously named duckduckgo-search before being renamed to reflect its
broader scope as a metasearch tool.

Citations:
  - https://pypi.org/project/ddgs/
```

Note the trajectory: search broadly, see promising snippet, fetch the one
authoritative result, synthesize. That's the pattern.

In [ ]:
# MEDIUM: requires synthesis across 2-3 sources
medium_q = (
    "What were the major changes announced in LangChain 1.0, "
    "released in October 2025?"
)
print(f"QUERY: {medium_q}")
print("=" * 70)
result_med = run_agent(medium_q, verbose=True)
print("=" * 70)
print(f"\n✓ Steps used: {result_med['steps']}")
print(f"✓ Citations: {len(result_med['citations'])}")
print(f"\nAnswer:\n{result_med['answer']}")
print("\nCitations:")
for c in result_med["citations"]:
    print(f"  - {c['title'][:60]} → {c['url']}")


**Sample output (yours will vary):**

```
QUERY: What were the major changes announced in LangChain 1.0...
======================================================================

── Step 1 ──
  → web_search({'query': 'LangChain 1.0 release October 2025', 'recency': 'year'}) → ok

── Step 2 ──
  → fetch_page({'url': 'https://blog.langchain.com/langchain-langgraph-1dot0/'}) → ok

── Step 3 ──
  → fetch_page({'url': 'https://docs.langchain.com/oss/python/migrate/langgraph-v1'}) → ok
  ◆ FINAL: LangChain 1.0 was released on October 22, 2025...
======================================================================

✓ Steps used: 3
✓ Citations: 2

Answer:
LangChain 1.0 was released on October 22, 2025 alongside LangGraph 1.0.
Major changes:
- A new create_agent abstraction with middleware support, superseding
  the legacy AgentExecutor and the LangGraph create_react_agent shortcut.
- Migration to TypedDict-based agent state...

Citations:
  - LangChain and LangGraph 1.0 Milestones → https://blog.langchain.com/langchain-langgraph-1dot0/
  - Migration guide - LangGraph v1 → https://docs.langchain.com/oss/python/migrate/langgraph-v1
```

The trajectory shows the pattern: one search, two fetches across
complementary sources (announcement blog + migration guide),
synthesis from both. Two citations because the agent read two pages.

## Step 6: Failure-mode walkthrough

This is the *heart* of the lab. Easy questions don't exercise the agent's
real intelligence. The failure-recovery patterns do.

We'll drive each failure mode synthetically and observe how the agent
handles it. The agent code doesn't change — only the test scenarios do.

In [ ]:
# Failure mode 1: EMPTY RESULTS
# Drive an extremely specific query that's unlikely to match anything.

print("─" * 70)
print("FAILURE MODE 1: Empty search results")
print("─" * 70)
empty_result = web_search(
    "xyzqq789nonexistentphrase asdkjhasdjkhasdkjh",
    max_results=5,
)
print(f"\nDirect tool call: {empty_result['status']}")
print(f"Detail: {empty_result.get('detail', '')}")

# The agent's response: when web_search returns 'empty', the agent
# should re-query with different terms, NOT loop forever, NOT fabricate.
# Let's see what it does:
print("\nAgent behavior with a deliberately obscure question:")
result_empty = run_agent(
    "Tell me about the AAAA-XYZ-9999 protocol designed in 1847",
    verbose=True,
)
print(f"\n  Stopped reason: {result_empty['stopped_reason']}")
print(f"  Answer (first 200 chars): {result_empty['answer'][:200]}")


**Sample output (yours will vary):**

```
──────────────────────────────────────────────────────────────────────
FAILURE MODE 1: Empty search results
──────────────────────────────────────────────────────────────────────

Direct tool call: empty
Detail: no results returned

Agent behavior with a deliberately obscure question:

── Step 1 ──
  → web_search({'query': 'AAAA-XYZ-9999 protocol 1847'}) → empty

── Step 2 ──
  → web_search({'query': '1847 communication protocol XYZ'}) → empty

── Step 3 ──
  ◆ FINAL: I could not find any reliable information about an...

  Stopped reason: answer_without_fetch
  Answer (first 200 chars): I could not find any reliable information about an
  "AAAA-XYZ-9999 protocol from 1847." This appears to be either a fictional
  or non-existent topic. If you have additional context...
```

The agent re-queried once, got empty again, and *correctly surfaced the
absence* rather than hallucinating a protocol that doesn't exist. This is
the "graceful degradation" behavior the system prompt is trying to elicit.

In [ ]:
# Failure mode 2: PAYWALL — fetch_page returns 'blocked'
# Many news sites serve paywalls; let's pick a known one to demonstrate.

print("─" * 70)
print("FAILURE MODE 2: Paywall detection")
print("─" * 70)
# Try fetching a likely-paywalled NYT or WSJ article URL. We use an
# example URL pattern; the exact behavior depends on the site's current
# state. The detection works on content-length + paywall words heuristic.

paywall_test = fetch_page(
    "https://www.nytimes.com/2024/01/15/technology/example.html"
)
print("\nDirect fetch_page on a likely-paywalled URL:")
print(f"  status: {paywall_test['status']}")
print(f"  kind: {paywall_test.get('kind', '')}")
print(f"  detail: {paywall_test.get('detail', '')}")


**Sample output (depends on the site's current state):**

```
──────────────────────────────────────────────────────────────────────
FAILURE MODE 2: Paywall detection
──────────────────────────────────────────────────────────────────────

Direct fetch_page on a likely-paywalled URL:
  status: error
  kind: blocked
  detail: HTTP 404
```

(Or `kind: blocked, detail: paywall or registration wall detected`,
depending on the URL.)

The key point: `fetch_page` doesn't crash on a paywall. It returns a
structured error with `kind=blocked`, which the agent can read and
respond to by trying a different result.

In [ ]:
# Failure mode 3: REPEATED ACTION
# If the model tries to do the same search twice, our loop refuses.
# We can demonstrate this directly by inspecting the loop's behavior:

print("─" * 70)
print("FAILURE MODE 3: Repeated-action detection")
print("─" * 70)
print("Simulated trajectory: the agent calls web_search twice with")
print("identical arguments. The second call returns a 'repeated_action'")
print("error, prompting the model to vary its query.\n")

seen = set()
for attempt in range(3):
    args = {"query": "ddgs python library", "recency": "any"}
    ah = _action_hash("web_search", args)
    if ah in seen:
        print(f"  Attempt {attempt+1}: REFUSED (repeated_action)")
    else:
        seen.add(ah)
        print(f"  Attempt {attempt+1}: would execute (new action)")


**Sample output:**

```
──────────────────────────────────────────────────────────────────────
FAILURE MODE 3: Repeated-action detection
──────────────────────────────────────────────────────────────────────
Simulated trajectory: the agent calls web_search twice with
identical arguments. The second call returns a 'repeated_action'
error, prompting the model to vary its query.

  Attempt 1: would execute (new action)
  Attempt 2: REFUSED (repeated_action)
  Attempt 3: REFUSED (repeated_action)
```

The protection isn't subtle, but it prevents the most common
infinite-loop failure: the model "thinks" the search didn't work and
just tries the same query again, expecting different results. That's a
classic agent failure mode in real systems.

## Step 7: Citations check

The whole point of citation tracking is that it's *trustworthy*. Let's
verify by inspecting one of our earlier query results closely.

In [ ]:
# Inspect the citations from the medium-difficulty question
print("─" * 70)
print("Citations from the medium query:")
print("─" * 70)
for i, c in enumerate(result_med.get("citations", []), 1):
    print(f"  {i}. {c['title']}")
    print(f"     {c['url']}")
print()
print(f"Total fetched: {len(result_med.get('citations', []))}")
print()
print("The URLs above are exactly the ones fetch_page returned ok/too_long")
print("for. The agent CANNOT have cited a URL it didn't actually fetch.")
print("Try grep'ing the answer text for these URLs and see if it mentions")
print("them — but understand the tracker is the source of truth, not the")
print("model's own claim.")


**Sample output:**

```
──────────────────────────────────────────────────────────────────────
Citations from the medium query:
──────────────────────────────────────────────────────────────────────
  1. LangChain and LangGraph 1.0 Milestones
     https://blog.langchain.com/langchain-langgraph-1dot0/
  2. Migration guide - LangGraph v1 - Docs by LangChain
     https://docs.langchain.com/oss/python/migrate/langgraph-v1

Total fetched: 2

The URLs above are exactly the ones fetch_page returned ok/too_long
for. The agent CANNOT have cited a URL it didn't actually fetch.
```

Citation trust is a *structural* property of how our loop works, not a
behavior we have to nag the model to follow. This is the right way to
build it.

## Step 8 (stretch): Swap the search backend to Tavily

`ddgs` is great for a lab, but for production you want a search backend
with an actual ToS. Tavily is the natural alternative — same interface
shape, just different field names. If you have a Tavily API key set in
`.env` as `TAVILY_API_KEY`, the swap is a few lines:

```python
def web_search_tavily(query, recency="any", max_results=8):
    from tavily import TavilyClient
    client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
    kwargs = {"max_results": max_results, "search_depth": "basic"}
    # Tavily's recency parameter is 'time_range' with the same vocabulary
    if recency != "any":
        kwargs["time_range"] = recency  # 'day'|'week'|'month'|'year'
    try:
        response = client.search(query=query, **kwargs)
    except Exception as e:
        return {"status": "error", "kind": "other", "detail": str(e)}
    raw = response.get("results", [])
    if not raw:
        return {"status": "empty", "query": query, "detail": "no results"}
    return {
        "status": "ok",
        "results": [
            {
                "title": r["title"],
                "url": r["url"],
                "snippet": r.get("content", ""),  # Tavily field name
            }
            for r in raw
        ],
    }
```

Then in your `execute_tool` dispatcher:

```python
USE_TAVILY = bool(os.getenv("TAVILY_API_KEY"))

def execute_tool(name, args):
    if name == "web_search":
        fn = web_search_tavily if USE_TAVILY else web_search
        return fn(...)
    ...
```

That's it. Same loop, same tools from the agent's perspective. The model
doesn't know or care which backend ran — that's the whole point of
keeping the contract stable.

## ✓ Lab complete

You've built a research agent that handles the real web — searches,
fetches, synthesizes, cites — without a framework, without embeddings,
without a vector store. Just two tools, one loop, and careful design.

The patterns you used here transfer directly:

| Pattern | Path 02 (Agentic RAG) | Path 03 (Multi-Agent) |
|---|---|---|
| Multi-step trajectories | ✓ | ✓ |
| Structured tool errors | ✓ | ✓ |
| Citation tracking | ✓ (over chunks) | ✓ (across agents) |
| Repeated-action detection | ✓ | ✓ (also: prevent loops) |
| Step cap with graceful exit | ✓ | ✓ |

### What to do next

- 🧠 **Take the quiz:** [`quizzes/foundations/multi-step-research-agent.md`](../../quizzes/foundations/multi-step-research-agent.md)
- 🧭 **Move on to Path 02** — Agentic RAG. The mental model from Lab 03
  transfers cleanly; just the corpus changes.
- 🔄 **Optional revisit:** rebuild Lab 03's agent in LangGraph (extension
  of Lab 05). The state, conditional edges, and checkpointer all map
  one-to-one.

### Things you might do to extend this lab

A few directions if you want to push further on your own:

1. **Multi-query refinement.** Add a `refine_search` tool that lets the
   model explicitly try variants of a failing query. (We get most of this
   from re-search, but an explicit tool can be cleaner.)
2. **Source-quality filtering.** Have the agent prefer `arxiv.org`,
   `*.gov`, `*.edu`, official documentation domains. Implementable as a
   re-ranking step after `web_search`.
3. **Parallel page fetches.** When the agent picks 3 URLs to read, fetch
   them concurrently with `concurrent.futures`. The model's reasoning is
   sequential; the I/O doesn't have to be.
4. **Content-aware truncation.** Instead of cutting at `max_chars`, find
   the section most relevant to the query (e.g., paragraphs containing
   query terms) and return *that*. This is the simplest possible
   in-context retrieval.
5. **Persistent citation log.** Save all citations across queries to a
   JSON file so a multi-session research project accumulates a verifiable
   bibliography.

Each of these is a 20-100 line addition. None of them changes the agent's
core shape.